In [ ]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib
import seaborn
import joblib

print("Everything is ready!")

In [ ]:
df = pd.read_csv("../data/blood_donation.csv")

df.head()

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

In [ ]:
df.info()

In [ ]:
df["Eligible_for_Donation"].value_counts()

In [ ]:
df["Eligible_for_Donation"].value_counts(normalize=True) * 100

In [ ]:
df.duplicated().sum()

In [ ]:
df.nunique()

In [ ]:
df.isnull().sum()

In [ ]:
df = df.drop(columns=["Medical_Condition"])

df.isnull().sum()

In [ ]:
df = df.drop(columns=[
    "Donor_ID",
    "Full_Name",
    "Contact_Number",
    "Email"
])

df.head()

In [ ]:
features = [
    "Gender",
    "Age",
    "Blood_Group",
    "City",
    "State",
    "Country",
    "Last_Donation_Date",
    "Total_Donations",
    "Weight_kg",
    "Hemoglobin_g_dL",
    "Donation_Center",
    "Registration_Date"
]

X = df[features]
y = df["Eligible_for_Donation"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
X.dtypes

In [ ]:
X.head()

In [ ]:
y.value_counts()

In [ ]:
X["Last_Donation_Date"] = pd.to_datetime(
    X["Last_Donation_Date"],
    dayfirst=True,
    format="mixed",
    errors="coerce"
)

print("Invalid Last Donation Dates:",
      X["Last_Donation_Date"].isna().sum())

In [ ]:
df.loc[
    pd.to_datetime(
        df["Last_Donation_Date"],
        dayfirst=True,
        format="mixed",
        errors="coerce"
    ).isna(),
    "Last_Donation_Date"
].head(20)

In [ ]:
df["Last_Donation_Date"].head(10).tolist()

In [ ]:
# Keep information about donors who never donated
X["Never_Donated"] = (
    X["Last_Donation_Date"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("never")
    .astype(int)
)

# Convert actual dates; "Never" becomes NaT
X["Last_Donation_Date"] = pd.to_datetime(
    X["Last_Donation_Date"],
    dayfirst=True,
    errors="coerce"
)

# Calculate days since last donation
X["Days_Since_Last_Donation"] = (
    pd.Timestamp.today().normalize() - X["Last_Donation_Date"]
).dt.days

# Remove the raw date columns
X = X.drop(columns=["Last_Donation_Date"])

In [ ]:
X[["Never_Donated", "Days_Since_Last_Donation"]].head(10)

In [ ]:
X["Never_Donated"].value_counts()

In [ ]:
X["Days_Since_Last_Donation"].describe()

In [ ]:
X["Never_Donated"].value_counts()

In [ ]:
df["Last_Donation_Date"].value_counts().head(30)

In [ ]:
df["Last_Donation_Date"].astype(str).str.contains(
    "never",
    case=False,
    na=False
).sum()

In [ ]:
X["Never_Donated"] = (
    df["Last_Donation_Date"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("never")
    .astype(int)
)

print(X["Never_Donated"].value_counts())

In [ ]:
X["Last_Donation_Date"] = pd.to_datetime(
    df["Last_Donation_Date"],
    dayfirst=True,
    errors="coerce"
)

X["Days_Since_Last_Donation"] = (
    pd.Timestamp.today().normalize() - X["Last_Donation_Date"]
).dt.days

In [ ]:
X["Registration_Date"] = pd.to_datetime(
    df["Registration_Date"],
    dayfirst=True,
    errors="coerce"
)

X["Registration_Year"] = X["Registration_Date"].dt.year

X = X.drop(columns=["Registration_Date"])

In [ ]:
X.dtypes

In [ ]:
df.info()

In [ ]:
df['Registration_Date'] = pd.to_datetime(df['Registration_Date'], errors='coerce')

df['Registration Year'] = df['Registration_Date'].dt.year

In [ ]:
df[['Registration_Date', 'Registration Year']].head()

In [ ]:
df['Registration Year'].isna().sum()

In [ ]:
df['Registration_Date'].head(10)

In [ ]:
df['Registration_Date'].head(10).to_list()

In [ ]:
df['Registration_Date'].value_counts(dropna=False).head(20)

In [ ]:
df = df.drop(
    columns=['Registration_Date', 'Registration Year', 'Destination Date'],
    errors='ignore'
)

In [ ]:
df.info()

In [ ]:
df = df.drop(columns=['Last_Donation_Date'])

In [ ]:
df.isnull().sum()

In [ ]:
X = df.drop('Eligible_for_Donation', axis=1)
y = df['Eligible_for_Donation']

In [ ]:
X = pd.get_dummies(X, drop_first=True)

In [ ]:
X.dtypes

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

In [ ]:
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

In [ ]:
model.fit(X_train, y_train)

In [ ]:
y_val_pred = model.predict(X_val)

In [ ]:
from sklearn.metrics import accuracy_score

val_accuracy = accuracy_score(y_val, y_val_pred)

print("Validation Accuracy:", val_accuracy)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_val, y_val_pred))

In [ ]:
y_test_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred))

In [ ]:
y_train_pred = model.predict(X_train)

train_accuracy = accuracy_score(y_train, y_train_pred)

print("Training Accuracy:", train_accuracy)

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

In [ ]:
model.fit(X_train, y_train)

In [ ]:
train_accuracy = model.score(X_train, y_train)
val_accuracy = model.score(X_val, y_val)
test_accuracy = model.score(X_test, y_test)

print("Training Accuracy:", train_accuracy)
print("Validation Accuracy:", val_accuracy)
print("Test Accuracy:", test_accuracy)

In [ ]:
y_test_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, y_test_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

In [ ]:
import os
import joblib

os.makedirs('model', exist_ok=True)

joblib.dump(model, 'model/random_forest_model.pkl')

In [ ]:
import joblib

joblib.dump(model, 'model/random_forest_model.pkl')